# Capítulo 4 — Regressão Linear e Aplicações Financeiras

Este capítulo usa regressão como ferramenta de estimação, diagnóstico e interpretação financeira. A sequência conceitual é guiada pelo Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander, sem reproduzir o texto da obra.

**Objetivos:**

- estimar OLS manualmente e comparar com `statsmodels`;
- interpretar resíduos, ANOVA, $R^2$, erros padrão, testes t e F;
- entender Gauss-Markov e as consequências de multicolinearidade, autocorrelação e heterocedasticidade;
- aplicar regressão a beta CAPM, hedge ratio e fatores financeiros.

A regressão descreve associações condicionais. Correlação mede associação linear sem direção; beta mede a inclinação da resposta de um ativo a um fator.

## Como ler este capítulo

A regressão será lida como uma cadeia: **problema financeiro**, **modelo matemático**, **derivação OLS**, **implementação manual**, **comparação com statsmodels**, **diagnóstico**, **interpretação econômica**, **Armadilhas** e **Exercício**. Um coeficiente só deve ser interpretado depois de verificar escala, resíduos, dependência entre regressores e hipótese de erro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.decomposition import PCA

from quantfinance.regression import (
    calculate_vif,
    capm_regression,
    minimum_variance_hedge_ratio,
    ols_numpy,
    regression_diagnostics,
)

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 4.1 — Regressão simples e OLS

O modelo linear simples é

$$y_i=\alpha+\beta x_i+\varepsilon_i.$$

OLS escolhe os coeficientes que minimizam a soma dos resíduos ao quadrado:

$$\min_\beta (y-X\beta)'(y-X\beta).$$

Com posto completo, a solução matricial é

$$\hat\beta=(X'X)^{-1}X'y.$$

Sob as hipóteses de Gauss-Markov — linearidade nos parâmetros, exogeneidade condicional, ausência de colinearidade perfeita e variância esférica para eficiência — OLS é BLUE: melhor estimador linear não viesado. Normalidade não é necessária para não-viesamento, mas ajuda na inferência exata em amostras pequenas.

**Interpretação:** $\alpha$ é o nível previsto quando $x=0$; $\beta$ é a mudança média prevista em $y$ para uma unidade adicional de $x$.

**Exercício:** simule um erro correlacionado com $x$ e observe que a interpretação causal de OLS deixa de ser válida.

In [ ]:
rng = np.random.default_rng(42)
x = rng.normal(0, 1, 300)
eps = rng.normal(0, 0.8, 300)
y = 0.5 + 1.3 * x + eps

X = sm.add_constant(x)
manual = ols_numpy(X, y)
model = sm.OLS(y, X).fit()

print("Coeficientes manuais:", manual.coefficients)
print("Coeficientes statsmodels:", model.params)
print("Erro padrão:", manual.standard_errors)
print("t-statistics:", manual.t_statistics)
print("R²:", manual.r_squared)
print("R² ajustado:", manual.adjusted_r_squared)
print("Estatística F:", manual.f_statistic)

plt.figure(figsize=(8, 4))
plt.scatter(x, y, alpha=0.35, label="observações")
order = np.argsort(x)
plt.plot(x[order], manual.fitted_values[order], color="black", label="reta OLS")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Regressão linear simples")
plt.show()

### OLS manual em álgebra matricial

Expandindo a soma de quadrados e igualando o gradiente a zero:

$$X'X\hat\beta=X'y.$$

Quando $X'X$ é invertível, obtemos a fórmula fechada. Em aplicações numéricas grandes, `solve` pode ser preferível a formar a inversa; aqui mostramos a inversa porque ela torna a álgebra do estimador explícita.

In [ ]:
beta_hat_formula = np.linalg.inv(X.T @ X) @ X.T @ y
print("beta = (X'X)^(-1)X'y:", beta_hat_formula)
print("beta ols_numpy:", manual.coefficients)
print("beta statsmodels:", model.params)
print("Equivalência NumPy/statsmodels:", np.allclose(beta_hat_formula, model.params))

## 4.2 — Resíduos, ANOVA, $R^2$, erros padrão e testes

O resíduo é $e_i=y_i-\hat y_i$. A decomposição ANOVA é

$$TSS=ESS+RSS,$$

onde $TSS=\sum(y_i-\bar y)^2$, $ESS=\sum(\hat y_i-\bar y)^2$ e $RSS=\sum e_i^2$.

$$R^2=1-\frac{RSS}{TSS}, \qquad
\bar R^2=1-(1-R^2)\frac{n-1}{n-k}.$$

O erro padrão quantifica incerteza do coeficiente. O teste t avalia um coeficiente individual; o teste F avalia a significância conjunta dos regressores. Um $R^2$ alto não prova causalidade nem garante previsão fora da amostra.

**Exercício:** compare $R^2$ e $R^2$ ajustado ao adicionar um regressor irrelevante.

In [ ]:
residuals = manual.residuals
rss = np.sum(residuals**2)
tss = np.sum((y - y.mean())**2)
ess = np.sum((manual.fitted_values - y.mean())**2)

anova_table = pd.DataFrame({
    "componente": ["ESS", "RSS", "TSS"],
    "soma_quadrados": [ess, rss, tss],
})
display(anova_table)
print("TSS = ESS + RSS:", np.isclose(tss, ess + rss))
print("R²:", manual.r_squared)
print("R² ajustado:", manual.adjusted_r_squared)
print("Erro padrão dos coeficientes:", manual.standard_errors)
print("t-statistics:", manual.t_statistics)
print("F-statistic:", manual.f_statistic)

plt.figure(figsize=(8, 4))
plt.scatter(manual.fitted_values, residuals, alpha=0.4)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Valores ajustados")
plt.ylabel("Resíduos")
plt.title("Resíduos versus valores ajustados")
plt.show()

## 4.3 — Regressão múltipla e aplicações multifatoriais

Na regressão múltipla,

$$y=\alpha+\beta_1x_1+\cdots+\beta_px_p+\varepsilon.$$

Cada coeficiente mede a associação parcial com $y$, mantendo os demais regressores constantes. Em finanças, fatores podem representar mercado, tamanho, valor, momentum ou choques macroeconômicos.

**Interpretação:** a significância de um fator depende da escala, da correlação com os demais fatores e da incerteza residual; não deve ser lida isoladamente.

**Exercício:** remova um fator do modelo e compare os coeficientes, $R^2$ ajustado e erro residual.

In [ ]:
x1 = rng.normal(size=500)
x2 = 0.7 * x1 + rng.normal(scale=0.7, size=500)
x3 = rng.normal(size=500)
y_multi = 0.2 + 0.8 * x1 - 0.4 * x2 + 0.6 * x3 + rng.normal(scale=0.8, size=500)

features = pd.DataFrame({"fator_mercado": x1, "fator_valor": x2, "fator_momentum": x3})
features_with_constant = sm.add_constant(features)
multifactor = sm.OLS(y_multi, features_with_constant).fit()

print(multifactor.summary())
print("Risco residual (desvio padrão):", multifactor.resid.std(ddof=1))

## 4.4 — Multicolinearidade, VIF e regressão por componentes

Multicolinearidade ocorre quando colunas de $X$ carregam informação redundante. Ela não necessariamente viesará OLS, mas aumenta a variância dos coeficientes e torna sinais e testes instáveis.

O VIF do regressor $j$ é

$$VIF_j=\frac{1}{1-R_j^2},$$

onde $R_j^2$ vem da regressão de $x_j$ contra os demais regressores.

Uma alternativa quando a prioridade é reduzir colinearidade é projetar os regressores em componentes ortogonais por PCA e regressar sobre esses componentes. A interpretação muda: os coeficientes passam a ser efeitos de fatores sintéticos, não dos ativos originais.

**Exercício:** aumente a correlação entre dois fatores e observe o crescimento do VIF e dos erros padrão.

In [ ]:
vif_values = calculate_vif(features.to_numpy())
vif_table = pd.DataFrame({"variável": features.columns, "VIF": vif_values})
display(vif_table)

pca = PCA().fit(features)
principal_components = pca.transform(features)
pc_model = sm.OLS(y_multi, sm.add_constant(principal_components)).fit()
print("Variância explicada pela PCA:", pca.explained_variance_ratio_)
print("R² da regressão nos componentes:", pc_model.rsquared)
print("Coeficientes nos componentes:", pc_model.params)

## 4.5 — Autocorrelação, heterocedasticidade e erros robustos

Autocorrelação significa que resíduos próximos no tempo se relacionam. Heterocedasticidade significa que a variância condicional dos resíduos muda com os regressores.

O Durbin-Watson é aproximadamente 2 quando não há autocorrelação de primeira ordem. Breusch-Godfrey testa autocorrelação de ordens maiores. White e Breusch-Pagan testam heterocedasticidade por construções diferentes.

Quando a variância não é constante, erros padrão robustos HC3 podem melhorar a inferência assintótica, mas não corrigem viés por endogeneidade. Autocorrelação também pode exigir HAC/Newey-West ou um modelo dinâmico.

**Exercício:** simule resíduos AR(1) e resíduos com variância crescente; compare os diagnósticos com o caso homocedástico.

In [ ]:
diagnostics = regression_diagnostics(multifactor)
display(pd.Series(diagnostics, name="valor").to_frame())

robust_multifactor = multifactor.get_robustcov_results(cov_type="HC3")
robust_table = pd.DataFrame({
    "coeficiente": multifactor.params,
    "erro_padrao_classico": multifactor.bse,
    "erro_padrao_HC3": robust_multifactor.bse,
    "t_HC3": robust_multifactor.tvalues,
})
display(robust_table)

# Exemplo explícito de resíduos autocorrelacionados e heterocedásticos.
time = np.arange(400)
ar_noise = np.zeros(400)
for index in range(1, 400):
    ar_noise[index] = 0.6 * ar_noise[index - 1] + rng.normal(scale=0.5 + 0.002 * index)
serial_target = 1.0 + 0.5 * np.sin(time / 25) + ar_noise
serial_model = sm.OLS(serial_target, sm.add_constant(time)).fit()
print("Diagnósticos série temporal:")
display(pd.Series(regression_diagnostics(serial_model)))

## 4.6 — WLS e GLS

WLS atribui pesos inversamente proporcionais à variância conhecida ou modelada:

$$\hat\beta_{WLS}=(X'WX)^{-1}X'Wy.$$

GLS usa uma matriz de covariância geral dos erros:

$$\hat\beta_{GLS}=(X'\Omega^{-1}X)^{-1}X'\Omega^{-1}y.$$

WLS é um caso particular de GLS quando $\Omega$ é diagonal. O ganho depende de especificar pesos/covariância de forma razoável; pesos incorretos podem piorar a eficiência.

**Exercício:** compare OLS e WLS quando a variância do erro cresce com $x$.

In [ ]:
n = 400
x_wls = np.linspace(0, 10, n)
sigma_wls = 0.5 + 0.2 * x_wls
rng = np.random.default_rng(42)
y_wls = 1 + 2 * x_wls + rng.normal(scale=sigma_wls)
X_wls = sm.add_constant(x_wls)

ols_model = sm.OLS(y_wls, X_wls).fit()
wls_model = sm.WLS(y_wls, X_wls, weights=1 / sigma_wls**2).fit()

print("OLS:", ols_model.params)
print("WLS:", wls_model.params)
print("Erro padrão OLS:", ols_model.bse)
print("Erro padrão WLS:", wls_model.bse)

# GLS com uma covariância AR(1) conhecida e consistente com os erros simulados.
correlation_parameter = 0.4
indices = np.arange(n)
ar_correlation = correlation_parameter ** np.abs(indices[:, None] - indices[None, :])
ar_scale = 0.5 + 0.002 * indices
omega = np.diag(ar_scale) @ ar_correlation @ np.diag(ar_scale)
ar_shocks = rng.multivariate_normal(np.zeros(n), omega)
y_gls = 1 + 2 * x_wls + ar_shocks
gls_model = sm.GLS(y_gls, X_wls, sigma=omega).fit()
print("GLS:", gls_model.params)

## 4.7 — Aplicações financeiras: CAPM, hedge e fatores

### CAPM

O CAPM em excesso de retorno é

$$R_i-R_f=\alpha+\beta(R_m-R_f)+\varepsilon.$$

- $\alpha$: retorno médio do ativo não explicado pelo mercado;
- $\beta$: exposição marginal ao fator de mercado;
- $R^2$: fração da variação amostral explicada pelo fator;
- $\varepsilon$: risco residual, específico do ativo.

O beta não é correlação. A relação é

$$\beta=\frac{\operatorname{Cov}(R_i,R_m)}{\operatorname{Var}(R_m)},$$

enquanto $\rho_{im}$ é a covariância dividida pelos dois desvios padrão. Portanto, $\beta=\rho_{im}\sigma_i/\sigma_m$.

### Hedge ratio

Para proteger uma variação spot com futuros, o hedge mínimo-variância é

$$h^*=\frac{\operatorname{Cov}(\Delta S,\Delta F)}{\operatorname{Var}(\Delta F)},$$

que coincide com a inclinação de uma regressão de $\Delta S$ em $\Delta F$ com intercepto.

### Regressão multifatorial

Um ativo pode ser explicado por vários fatores:

$$R_i-R_f=\alpha+\beta_1F_1+\beta_2F_2+\cdots+\varepsilon.$$

**Exercício:** altere a volatilidade residual do CAPM e observe o efeito no $R^2$ e no risco específico.

In [ ]:
risk_free = 0.0001
market = rng.normal(0.0004, 0.012, 1_000)
asset = risk_free + 0.0002 + 1.25 * (market - risk_free) + rng.normal(0, 0.008, 1_000)

capm_manual = capm_regression(asset, market, risk_free_rate=risk_free)
capm_reference = sm.OLS(
    asset - risk_free,
    sm.add_constant(market - risk_free),
).fit()
market_excess = market - risk_free
asset_excess = asset - risk_free
beta_covariance = np.cov(asset_excess, market_excess, ddof=1)[0, 1] / np.var(market_excess, ddof=1)
asset_beta = capm_manual.coefficients[1]
residual_risk = capm_manual.residuals.std(ddof=1)
correlation = np.corrcoef(asset_excess, market_excess)[0, 1]

print("Alpha manual:", capm_manual.coefficients[0])
print("Beta manual:", asset_beta)
print("Beta statsmodels:", capm_reference.params[1])
print("Beta pela covariância/variância:", beta_covariance)
print("Correlação ativo/mercado:", correlation)
print("R² CAPM:", capm_manual.r_squared)
print("Risco residual:", residual_risk)
print("Beta não é correlação:", not np.isclose(asset_beta, correlation))

futures = rng.normal(0, 0.01, 1_000)
spot_changes = 0.85 * futures + rng.normal(0, 0.006, 1_000)
hedge_ratio = minimum_variance_hedge_ratio(spot_changes, futures)
hedge_reference = sm.OLS(spot_changes, sm.add_constant(futures)).fit()
print("Hedge ratio por covariância:", hedge_ratio)
print("Hedge ratio por regressão:", hedge_reference.params[1])

factor_data = pd.DataFrame({
    "mercado": market_excess,
    "valor": rng.normal(0, 0.01, 1_000),
    "momentum": rng.normal(0, 0.008, 1_000),
})
multifactor_asset = (
    0.0001
    + 1.1 * factor_data["mercado"]
    + 0.4 * factor_data["valor"]
    - 0.2 * factor_data["momentum"]
    + rng.normal(0, 0.006, 1_000)
)
multifactor_reference = sm.OLS(
    multifactor_asset,
    sm.add_constant(factor_data),
).fit()
print("\nRegressão multifatorial:")
print(multifactor_reference.params)
print("R² multifatorial:", multifactor_reference.rsquared)

## Exercícios integradores

1. Derive a fórmula de OLS a partir da minimização da soma dos quadrados.
2. Compare os coeficientes, erros padrão, $R^2$ e teste F do OLS manual com `statsmodels`.
3. Simule multicolinearidade e interprete os VIFs.
4. Compare os coeficientes originais com uma regressão sobre componentes principais.
5. Gere resíduos AR(1) e resíduos heterocedásticos e aplique os diagnósticos.
6. Compare OLS, erros HC3, WLS e GLS no mesmo conjunto de dados.
7. Calcule beta por OLS, por covariância/variância e compare com correlação.
8. Explique alpha, beta, $R^2$ e risco residual no CAPM.
9. Estime um hedge ratio e compare a fórmula de covariância com a inclinação da regressão.
10. Adicione ou remova um fator na regressão multifatorial e analise a mudança no risco residual.

## Leitura crítica dos resultados

Leia um coeficiente junto com escala, erro padrão, resíduos e desenho da amostra. $R^2$ alto não garante causalidade; p-valor baixo não corrige endogeneidade; erro robusto não corrige viés de variável omitida. Antes de interpretar beta ou alpha, confirme a unidade dos retornos e a definição do intercepto.